In [9]:
import os
import time
import random
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report


In [10]:
# تنظیم Seed جهت Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 Computing Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

🔥 Computing Device: cuda
GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU


In [11]:
class WatermarkDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['image_path']
        label = self.df.iloc[idx]['label']

        # لود ایمن و یکسان‌سازی فرمت رنگی
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)




In [12]:

# 1. مسیر اصلی داده‌ها (Raw String برای جلوگیری از خطای Escape ویندوز)
DATASET_ROOT = Path(r"D:\olampic_fanavari\watermark\dk-dataset\dataset")
TRAIN_DIR = DATASET_ROOT / "train"
TEST_DIR = DATASET_ROOT / "test"

# 2. اسکن فایل‌ها و ساخت متادیتا فریم
def scan_train_data(train_path: Path) -> pd.DataFrame:
    records = []
    # مپ کردن نام پوشه‌ها به برچسب‌های باینری
    class_mapping = {
        "negative": 0,
        "positive": 1
    }
    
    for folder_name, label in class_mapping.items():
        folder_path = train_path / folder_name
        if not folder_path.exists():
            # بررسی حالت حساس به حروف کوچک/بزرگ ویندوزی
            matched_dirs = [d for d in train_path.iterdir() if d.name.lower() == folder_name]
            if matched_dirs:
                folder_path = matched_dirs[0]
            else:
                continue
                
        for img_file in folder_path.iterdir():
            if img_file.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]:
                records.append({
                    "image_path": str(img_file),
                    "label": label,
                    "class_name": folder_name
                })
                
    df = pd.DataFrame(records)
    return df

df_all = scan_train_data(TRAIN_DIR)
print(f"Total images found: {len(df_all)}")
print(df_all['class_name'].value_counts())

# 3. تقسیم تمیز به Train و Validation با حفظ تعادل کلاس‌ها (Stratified)
train_df, val_df = train_test_split(
    df_all,
    test_size=0.15,
    random_state=42,
    stratify=df_all["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"\nTrain set: {len(train_df)} images")
print(f"Validation set: {len(val_df)} images")


Total images found: 8582
class_name
negative    4291
positive    4291
Name: count, dtype: int64

Train set: 7294 images
Validation set: 1288 images


In [13]:
train_transforms = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(
    WatermarkDataset(train_df, transform=train_transforms),
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    WatermarkDataset(val_df, transform=val_transforms),
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("DataLoaders are ready!")
print(f"تعداد داده‌های آموزش: {len(train_loader)} | داده‌های اعتبارسنجی: {len(val_loader)}")


DataLoaders are ready!
تعداد داده‌های آموزش: 228 | داده‌های اعتبارسنجی: 41


In [14]:
def build_convnext_detector():
    weights = models.ConvNeXt_Tiny_Weights.DEFAULT
    model = models.convnext_tiny(weights=weights)
    
    # فعال‌سازی کامل گرادیان‌ها برای Fine-Tuning انتخابی
    for param in model.parameters():
        param.requires_grad = True
        
    in_features = model.classifier[2].in_features
    # جایگزینی هد با Dropout برای مقابله با اورفیتینگ روی پس‌زمینه‌های یکنواخت
    model.classifier[2] = nn.Sequential(
        nn.Dropout(p=0.35),
        nn.Linear(in_features, 1)
    )
    return model

model = build_convnext_detector().to(device)


In [15]:
model

ConvNeXt(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=96, out_features=384, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=384, out_features=96, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=

In [16]:
import torch

print("--- 1. تست ساده CUDA ---")
x = torch.randn(10, 10, device="cuda")
print("CUDA اوکی است:", x.is_cuda)

print("\n--- 2. تست لود یک بچ خام (بدون حلقه آموزش) ---")
try:
    first_batch = next(iter(train_loader))
    imgs, lbls = first_batch
    print(f"بچ اول با موفقیت خوانده شد! شیپ تصاویر: {imgs.shape}, شیپ لیبل‌ها: {lbls.shape}")
except Exception as e:
    print(f"خطا در خواندن داده: {e}")

print("\n--- 3. تست فوروارد پس روی GPU ---")
try:
    imgs = imgs.to("cuda")
    # فرض بر اینکه مدل شما به نام model تعریف شده است
    model.eval()
    with torch.no_grad():
        out = model(imgs)
    print("فوروارد پس موفق بود! شیپ خروجی:", out.shape)
except Exception as e:
    print(f"خطا در مدل یا فوروارد پس: {e}")


--- 1. تست ساده CUDA ---
CUDA اوکی است: True

--- 2. تست لود یک بچ خام (بدون حلقه آموزش) ---
بچ اول با موفقیت خوانده شد! شیپ تصاویر: torch.Size([32, 3, 384, 384]), شیپ لیبل‌ها: torch.Size([32])

--- 3. تست فوروارد پس روی GPU ---
فوروارد پس موفق بود! شیپ خروجی: torch.Size([32, 1])


In [ ]:
import time
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# پاکسازی کش GPU قبل از شروع
torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

EPOCHS = 7
optimizer = torch.optim.AdamW([
    {'params': model.features.parameters(), 'lr': 2e-5},
    {'params': model.classifier.parameters(), 'lr': 2e-4}
], weight_decay=1e-2)

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)

history = {
    'train_loss': [], 'val_loss': [],
    'accuracy': [], 'f1_score': [],
    'precision': [], 'recall': []
}

best_val_f1 = 0.0

print(f"🔥 شروع آموزش روی: {torch.cuda.get_device_name(0)}")

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # ---------- مرحله آموزش ----------
    model.train()
    running_loss = 0.0
    
    # mininterval=1.0 از اسپم کردن کنسول ویندوز جلوگیری میکند
    train_bar = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}] Train", mininterval=0.5)
    
    for batch_idx, (images, labels) in enumerate(train_bar):
        # حذف non_blocking جهت جلوگیری از synchronization deadlock
        images = images.to(device)
        labels = labels.to(device, dtype=torch.float32).unsqueeze(1)
        
        optimizer.zero_grad(set_to_none=True) # بهینه‌تر از zero_grad ساده
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        running_loss += loss.item()
        train_bar.set_postfix({"loss": f"{loss.item():.4f}"})
        
    train_loss = running_loss / len(train_loader)
    scheduler.step()
    
    # ---------- مرحله ارزیابی ----------
    model.eval()
    val_loss = 0.0
    val_preds, val_targets = [], []
    
    val_bar = tqdm(val_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}] Val", mininterval=0.5)
    with torch.no_grad():
        for images, labels in val_bar:
            images = images.to(device)
            labels = labels.to(device, dtype=torch.float32).unsqueeze(1)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            probs = torch.sigmoid(outputs).cpu().squeeze(1).numpy()
            val_preds.extend((probs >= 0.5).astype(int))
            val_targets.extend(labels.cpu().squeeze(1).numpy().astype(int))
            
    val_loss = val_loss / len(val_loader)
    acc = accuracy_score(val_targets, val_preds)
    f1 = f1_score(val_targets, val_preds, zero_division=0)
    precision = precision_score(val_targets, val_preds, zero_division=0)
    recall = recall_score(val_targets, val_preds, zero_division=0)
    
    # ذخیره در تاریخچه
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['accuracy'].append(acc)
    history['f1_score'].append(f1)
    history['precision'].append(precision)
    history['recall'].append(recall)
    
    elapsed = time.time() - start_time
    print(f"\n📊 [Epoch {epoch+1}/{EPOCHS} - {elapsed:.1f}s] "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Acc: {acc:.4f} | F1: {f1:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")
    
    if f1 > best_val_f1:
        best_val_f1 = f1
        torch.save(model.state_dict(), "best_watermark_model.pth")
        print(f"🎯 چک‌پوینت جدید ذخیره شد! (F1: {best_val_f1:.4f})")
    print("-" * 75)

print(f"\n🚀 آموزش تمام شد. بالاترین F1 اعتبارسنجی: {best_val_f1:.4f}")


c:\Users\Acer\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔥 شروع آموزش روی: NVIDIA GeForce RTX 4060 Laptop GPU


Epoch [1/7] Train:   0%|          | 0/228 [00:00<?, ?it/s]